Evolutionary Path Planner

Can Akkus

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
rng = random.Random(SEED)

In [ ]:
def make_grid(size=10, p_obstacle=0.2, rng=None):
    assert rng is not None

    grid = [[0 for _ in range(size)] for _ in range(size)]
    for y in range(size):
        for x in range(size):
            if (x, y) == (0, 0) or (x, y) == (size-1, size-1):
                grid[y][x] = 0
            else:
                grid[y][x] = 1 if rng.random() < p_obstacle else 0
    return grid

GRID_SIZE = 10
GOAL = (GRID_SIZE - 1, GRID_SIZE - 1)

grid = make_grid(size=GRID_SIZE, p_obstacle=0.2, rng=rng)

In [ ]:
obstacles = sum(cell == 1 for row in grid for cell in row)
print("Obstacle count:", obstacles)
print("Start cell:", grid[0][0])
print("Goal cell:", grid[GOAL[1]][GOAL[0]])

Reusing Part A Components

In Part A, the core building blocks of an Evolutionary Algorithm were implemented and tested. These components are reused here without modification to construct a complete evolutionary path planner.


In [ ]:
def random_individual(length: int, rng) -> list[int]:
    assert isinstance(length, int)
    assert length > 0
    assert rng is not None
    return [rng.randint(0, 3) for _ in range(length)]


def mutate_path(path, p, rng) -> list[int]:
    assert rng is not None
    new_path = path.copy()

    for i in range(len(new_path)):
        if rng.random() < p:
            new_path[i] = rng.randint(0, 3)

    return new_path


def crossover_path(p1: list[int], p2: list[int], rng):
    assert rng is not None

    if len(p1) != len(p2):
        raise ValueError("Parents must have the same length")

    length = len(p1)
    crossover_point = rng.randint(1, length - 1)

    child1 = p1[:crossover_point] + p2[crossover_point:]
    child2 = p2[:crossover_point] + p1[crossover_point:]

    return child1, child2


def tournament_select(population, k, fit_func, rng):
    assert rng is not None

    k = min(k, len(population))
    contestants = rng.sample(population, k)

    best = contestants[0]
    best_fitness = fit_func(best)

    for ind in contestants[1:]:
        f = fit_func(ind)
        if f > best_fitness:
            best_fitness = f
            best = ind

    return best


def simulate_path(genome: list[int], grid):
    x, y = 0, 0
    collisions = 0

    moves = {
        0: (0, -1),
        1: (0, 1),
        2: (-1, 0),
        3: (1, 0)
    }

    for move in genome:
        dx, dy = moves[move]
        nx, ny = x + dx, y + dy

        if nx < 0 or ny < 0 or ny >= len(grid) or nx >= len(grid[0]):
            collisions += 1
        elif grid[ny][nx] == 1:
            collisions += 1
        else:
            x, y = nx, ny

    return (x, y), collisions


def fitness(genome, grid, goal=(9,9)):
    (x, y), c = simulate_path(genome, grid)
    xg, yg = goal
    d = abs(x - xg) + abs(y - yg)

    return 1 / (d + c + 1)


B1 — Baseline Evolutionary Algorithm

In this section, the Part A components are combined into a full Evolutionary Algorithm (EA). The EA evolves fixed-length paths on a 10×10 grid with obstacles. Each generation includes evaluation, tournament selection, crossover, mutation, and elitism.

In [ ]:
N = 100
PATH_LEN = 25        
GENERATIONS = 60     
K = 3               
P_MUT = 0.1         

In [ ]:
def run_ea(rng, grid, goal):
    population = [random_individual(PATH_LEN, rng) for _ in range(N)]

    best_history = []
    mean_history = []

    best_individual = None
    best_fitness = -1

    for _ in range(GENERATIONS):
        fitness_values = [fitness(ind, grid, goal) for ind in population]

        best_index = int(np.argmax(fitness_values))
        best_of_gen = float(fitness_values[best_index])
        mean_of_gen = float(np.mean(fitness_values))

        best_history.append(best_of_gen)
        mean_history.append(mean_of_gen)

        if best_of_gen > best_fitness:
            best_fitness = best_of_gen
            best_individual = population[best_index][:]

        new_population = [population[best_index][:]]

        while len(new_population) < N:
            parent1 = tournament_select(population, K, lambda x: fitness(x, grid, goal), rng)
            parent2 = tournament_select(population, K, lambda x: fitness(x, grid, goal), rng)

            child1, child2 = crossover_path(parent1, parent2, rng)
            child1 = mutate_path(child1, P_MUT, rng)
            child2 = mutate_path(child2, P_MUT, rng)

            new_population.append(child1)
            if len(new_population) < N:
                new_population.append(child2)

        population = new_population

    return best_individual, best_fitness, best_history, mean_history

In [ ]:
best_ind, best_fit, best_hist, mean_hist = run_ea(rng, grid, GOAL)
print("Best fitness:", best_fit)

In [ ]:
plt.figure()
plt.plot(best_hist, label="Best fitness")
plt.plot(mean_hist, label="Mean fitness")
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title("EA Progress (Best vs Mean)")
plt.legend()
plt.show()

Interpretation

The best fitness increases rapidly in the first generations, showing that the evolutionary algorithm quickly finds better paths. After around ten generations, the best fitness reaches a plateau, which indicates that a near-optimal solution has been found. The mean fitness increases more slowly, meaning that not all individuals are equally good. This shows that diversity is preserved while the best solution is maintained through elitism.


B2 — Visualisation of the Best Evolved Path

The best individual found by the EA is visualised on the grid. The figure shows obstacles, start, goal, and the trajectory.

In [ ]:
def path_positions(genome, grid):
    x, y = 0, 0
    positions = [(x, y)]

    moves = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}

    for move in genome:
        dx, dy = moves[move]
        nx, ny = x + dx, y + dy

        if nx < 0 or ny < 0 or ny >= len(grid) or nx >= len(grid[0]):
            positions.append((x, y))
            continue

        if grid[ny][nx] == 1:
            positions.append((x, y))
            continue

        x, y = nx, ny
        positions.append((x, y))

    return positions

In [ ]:
def plot_path_on_grid(grid, positions, goal):
    xs = [p[0] for p in positions]
    ys = [p[1] for p in positions]

    grid_img = np.array(grid)

    plt.figure()
    plt.imshow(grid_img, origin="upper")
    plt.title("Best Evolved Path")
    plt.xlabel("x")
    plt.ylabel("y")

    plt.scatter([0], [0], marker="o", s=80, label="Start")
    plt.scatter([goal[0]], [goal[1]], marker="X", s=100, label="Goal")

    plt.plot(xs, ys, linewidth=2, label="Trajectory")
    plt.legend()
    plt.show()

positions = path_positions(best_ind, grid)
plot_path_on_grid(grid, positions, GOAL)

Commentary

The visualisation shows how the best evolved genome moves across the grid. In general, the path moves towards the goal while avoiding obstacles. Some repeated positions can happen when the robot attempts invalid moves and stays in place.


B3 — Parameter Sensitivity Study

In this section, the influence of different EA parameters is analysed. Each configuration is evaluated over multiple runs, while keeping all other parameters fixed.


In [ ]:
def run_ea_best_only(rng, grid, goal):
    population = [random_individual(PATH_LEN, rng) for _ in range(N)]
    best_history = []

    for _ in range(GENERATIONS):
        fitness_values = [fitness(ind, grid, goal) for ind in population]
        best_index = int(np.argmax(fitness_values))
        best_history.append(float(fitness_values[best_index]))

        new_population = [population[best_index][:]]

        while len(new_population) < N:
            p1 = tournament_select(population, K, lambda x: fitness(x, grid, goal), rng)
            p2 = tournament_select(population, K, lambda x: fitness(x, grid, goal), rng)

            c1, c2 = crossover_path(p1, p2, rng)
            c1 = mutate_path(c1, P_MUT, rng)
            c2 = mutate_path(c2, P_MUT, rng)

            new_population.append(c1)
            if len(new_population) < N:
                new_population.append(c2)

        population = new_population

    return best_history

In [ ]:
mutation_values = [0.02, 0.05, 0.1]

plt.figure()

for p in mutation_values:
    P_MUT = p
    all_runs = []

    for i in range(10):
        local_rng = random.Random(SEED + i)
        best_hist = run_ea_best_only(local_rng, grid, GOAL)
        all_runs.append(best_hist)

    avg_best = np.mean(all_runs, axis=0)
    plt.plot(avg_best, label=f"p = {p}")

plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Effect of Mutation Rate")
plt.legend()
plt.show()

In [ ]:
population_values = [10, 20, 40]

plt.figure()

for pop in population_values:
    N = pop
    all_runs = []

    for i in range(10):
        local_rng = random.Random(SEED + i)
        best_hist = run_ea_best_only(local_rng, grid, GOAL)
        all_runs.append(best_hist)

    avg_best = np.mean(all_runs, axis=0)
    plt.plot(avg_best, label=f"N = {pop}")

plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Effect of Population Size")
plt.legend()
plt.show()

In [ ]:
tournament_values = [2, 3, 5]

plt.figure()

for t in tournament_values:
    K = t
    all_runs = []

    for i in range(10):
        local_rng = random.Random(SEED + i)
        best_hist = run_ea_best_only(local_rng, grid, GOAL)
        all_runs.append(best_hist)

    avg_best = np.mean(all_runs, axis=0)
    plt.plot(avg_best, label=f"k = {t}")

plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Effect of Tournament Size")
plt.legend()
plt.show()

Analysis

Mutation rate strongly affects exploration. Lower mutation values lead to faster convergence but increase the risk of getting stuck. Higher mutation values preserve diversity but slow down convergence.

Population size mainly influences stability. Smaller populations show more fluctuation, while larger populations produce smoother curves.

Tournament size controls selection pressure. Larger tournaments increase exploitation but reduce diversity.
